# 방한 외래객 목적별 입국자 수 SARIMA 예측 (수정본)

원본 노트북(`original/Project.ipynb`)의 분석 흐름을 그대로 따르면서 아래 문제를 고친 버전입니다.

| 원본의 문제 | 수정 내용 |
|---|---|
| 유학연수(edu)·기타(etc) 셀에서 변수가 섞여, edu 결과 칸에 etc 결과가 들어감 | 5개 목적을 **하나의 반복문**으로 처리해 변수 혼동이 생기지 않게 함 |
| `pm.auto_arima(..., exog=...)`의 `exog` 인자가 무시되어 환율이 반영되지 않음 | pmdarima 인자 이름인 `X=`로 전달 |
| 차수 탐색 모델(auto_arima)과 최종 모델(SARIMAX)의 추정 설정이 달라 AIC를 직접 비교할 수 없었음 | **같은 차수·같은 설정**으로 SARIMA(환율 없음)와 SARIMAX(환율 포함)를 각각 적합해 AIC와 예측 오차를 비교 |
| 오차(MAE·RMSE·SMAPE)를 로그 값으로만 계산 | 로그 기준과 **실제 인원 기준(MAE·RMSE·MAPE)** 을 함께 계산 |
| 코로나 구간 더미 변수가 모두 0이라 의미가 없음 | 코로나 기간(2020~2022)을 데이터에서 명시적으로 제외하고 더미 제거 |

**데이터 구성**: 1998~2019년(코로나 이전)으로 학습하고, 2023~2024년(코로나 이후) 실제값과 비교합니다.
코로나 3년(36개월)을 제외하고 2019년 12월 다음을 2023년 1월로 이어 붙였습니다. 36개월은 12의 배수라 월별 계절 위상은 그대로 유지됩니다.

## 0. 환경 준비 (Colab)

In [ ]:
# Colab 에서 실행할 때만 필요합니다. 저장소를 받아 데이터 경로를 맞추고 pmdarima 를 설치합니다.
# !git clone https://github.com/song0202-g/inbound-tourism-forecast.git
# %cd inbound-tourism-forecast
!pip install -q pmdarima

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.statespace.sarimax import SARIMAX

warnings.filterwarnings("ignore")
plt.rcParams["figure.dpi"] = 110

## 1. 설정

In [ ]:
# 원본 분석에 쓴 1995~2025 파일이 있으면 그것을, 없으면 저장소에 포함된 2015~2025 파일을 사용합니다.
ENTRY_CSV_FULL = "data/inbound_by_purpose_1995_2025.csv"
ENTRY_CSV_SHORT = "data/inbound_by_purpose_2015_2025.csv"
ENTRY_CSV = ENTRY_CSV_FULL if os.path.exists(ENTRY_CSV_FULL) else ENTRY_CSV_SHORT
FX_CSV = "data/monthly_exchange_rate_98_24.csv"

START = "1998-01-01"              # 환율 데이터가 1998년부터 있음
TRAIN_END = "2019-12-01"          # 학습: 코로나 이전
COVID = ("2020-01-01", "2022-12-01")
TEST = ("2023-01-01", "2024-12-01")  # 평가: 코로나 이후 24개월

TARGETS = {"tour": "관광", "business": "상용", "public": "공용", "edu": "유학연수", "etc": "기타"}

FIG_DIR = "figures"
RESULT_DIR = "results"
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(RESULT_DIR, exist_ok=True)
print("입국 데이터:", ENTRY_CSV)

## 2. 데이터 불러오기 · 전처리

In [ ]:
raw = pd.read_csv(ENTRY_CSV, encoding="utf-8-sig")   # BOM 이 있어도 첫 열 이름이 깨지지 않게
raw = raw.rename(columns={"기준년월": "Date", **{v: k for k, v in TARGETS.items()}})
raw["Date"] = pd.to_datetime(raw["Date"].astype(str), format="%Y%m")

# 환율 날짜(1998.10 등)를 문자열로 읽어 10월이 1월로 바뀌는 문제를 피함
fx = pd.read_csv(FX_CSV, dtype={"Date": str})
fx["Date"] = pd.to_datetime(fx["Date"].str.strip(), format="%Y.%m")

data = (raw.merge(fx[["Date", "Avg_Rate"]], on="Date", how="inner")
           .set_index("Date").sort_index())
data = data.loc[data.index >= START, list(TARGETS) + ["Avg_Rate"]]

# 코로나 기간 제외 (원본 파일에 이미 빠져 있어도 안전하게 한 번 더 제외)
covid_mask = (data.index >= COVID[0]) & (data.index <= COVID[1])
data = data.loc[~covid_mask]
print("결측치:\n", data.isnull().sum().to_string())

log_data = np.log(data)   # 입국자 수와 환율 모두 로그 변환
train = log_data.loc[log_data.index <= TRAIN_END]
test = log_data.loc[(log_data.index >= TEST[0]) & (log_data.index <= TEST[1])]

print(f"학습: {train.index[0]:%Y-%m} ~ {train.index[-1]:%Y-%m} ({len(train)}개월)")
print(f"평가: {test.index[0]:%Y-%m} ~ {test.index[-1]:%Y-%m} ({len(test)}개월)")
if len(train) < 120:
    print("⚠️ 학습 기간이 10년 미만입니다. 1995~2025 원본 파일을 data/ 에 넣으면 원래 분석과 같은 조건이 됩니다.")
data.tail()

## 3. 시계열 구조 진단 (ACF · PACF · 차분)

In [ ]:
def plot_acf_pacf(series, title, path=None, lags=48):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    plot_acf(series.dropna(), lags=lags, ax=axes[0]); axes[0].set_title(f"{title} ACF")
    plot_pacf(series.dropna(), lags=lags, ax=axes[1], method="ywm"); axes[1].set_title(f"{title} PACF")
    fig.tight_layout()
    if path:
        fig.savefig(path, bbox_inches="tight")
    plt.show()


for t in TARGETS:
    s = log_data[t]
    plot_acf_pacf(s, t, f"{FIG_DIR}/{t}_acf_raw.png")                                  # 원계열
    plot_acf_pacf(s.diff(), f"{t} (d=1)", f"{FIG_DIR}/{t}_acf_d1.png")                   # 1차 차분
    plot_acf_pacf(s.diff().diff(12), f"{t} (d=1, D=1)", f"{FIG_DIR}/{t}_acf_d1D1.png")   # + 계절 차분

## 4. 차수 탐색 · 모델 적합 · 평가

목적별로 다음을 반복합니다.
1. `auto_arima`(AIC 기준, d=1, D=1, m=12)로 차수 탐색. 환율은 `X=`로 전달
2. 같은 차수·같은 설정으로 **SARIMA(환율 없음)** 와 **SARIMAX(환율 포함)** 적합 → AIC 비교, 환율 계수와 p-value 확인
3. 두 모델로 2023~2024년 예측 → 로그 기준 / 실제 인원 기준 오차 계산

평가 구간의 환율은 실제 관측값을 사용했습니다(사후 검증).

In [ ]:
def search_order(y, X):
    # pmdarima 가 있으면 auto_arima, 없으면 작은 격자 탐색으로 AIC 최소 차수를 찾습니다.
    try:
        import pmdarima as pm
        m = pm.auto_arima(y, X=X, d=1, D=1, m=12, seasonal=True,
                          start_p=1, start_q=1, max_p=3, max_q=3,
                          stepwise=True, suppress_warnings=True, error_action="ignore")
        return tuple(m.order), tuple(m.seasonal_order)
    except ImportError:
        best = None
        for p in range(4):
            for q in range(4):
                for P in range(2):
                    for Q in range(2):
                        if p + q + P + Q > 5:
                            continue
                        try:
                            r = fit_sarimax(y, X, (p, 1, q), (P, 1, Q, 12))
                        except Exception:
                            continue
                        if best is None or r.aic < best[0]:
                            best = (r.aic, (p, 1, q), (P, 1, Q, 12))
        return best[1], best[2]


def fit_sarimax(y, X, order, seasonal_order):
    return SARIMAX(y, exog=X, order=order, seasonal_order=seasonal_order,
                   enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)


def errors(y_log, f_log):
    # 로그 기준 MAE · RMSE · SMAPE, 실제 인원 기준 MAE · RMSE · MAPE
    y_log, f_log = np.asarray(y_log), np.asarray(f_log)
    y, f = np.exp(y_log), np.exp(f_log)
    return {
        "MAE(log)": np.mean(np.abs(y_log - f_log)),
        "RMSE(log)": np.sqrt(np.mean((y_log - f_log) ** 2)),
        "SMAPE(log,%)": 100 * np.mean(2 * np.abs(y_log - f_log) / (np.abs(y_log) + np.abs(f_log))),
        "MAE(명)": np.mean(np.abs(y - f)),
        "RMSE(명)": np.sqrt(np.mean((y - f) ** 2)),
        "MAPE(%)": 100 * np.mean(np.abs(y - f) / y),
    }


rows, forecasts = [], {}
for t, name in TARGETS.items():
    y_tr, y_te = train[t].values, test[t].values
    X_tr, X_te = train[["Avg_Rate"]].values, test[["Avg_Rate"]].values

    order, sorder = search_order(y_tr, X_tr)
    res_sarima = fit_sarimax(y_tr, None, order, sorder)
    res_sarimax = fit_sarimax(y_tr, X_tr, order, sorder)

    i = res_sarimax.model.param_names.index("x1")      # 환율 계수 위치
    fx_coef, fx_p = res_sarimax.params[i], res_sarimax.pvalues[i]

    for label, res, X_future in [("SARIMA", res_sarima, None), ("SARIMAX(환율)", res_sarimax, X_te)]:
        fc = res.get_forecast(steps=len(y_te), exog=X_future)
        mean, ci = fc.predicted_mean, fc.conf_int(alpha=0.05)
        forecasts[(t, label)] = (mean, ci)
        rows.append({"목적": f"{name}({t})", "모델": label, "차수": f"{order}x{sorder}",
                     "AIC": res.aic,
                     "환율 계수": fx_coef if label != "SARIMA" else np.nan,
                     "환율 p-value": fx_p if label != "SARIMA" else np.nan,
                     **errors(y_te, mean)})
    print(f"[{name}] {order}x{sorder} | AIC SARIMA {res_sarima.aic:.1f} vs SARIMAX {res_sarimax.aic:.1f} "
          f"| 환율 계수 {fx_coef:.3f} (p={fx_p:.3f})")

summary = pd.DataFrame(rows)
summary.to_csv(f"{RESULT_DIR}/summary.csv", index=False, encoding="utf-8-sig")
summary.round(3)

## 5. 예측 그래프

In [ ]:
for t, name in TARGETS.items():
    # 두 모델 중 AIC 가 낮은 쪽을 그림
    part = summary[summary["목적"] == f"{name}({t})"]
    best_label = part.loc[part["AIC"].idxmin(), "모델"]
    mean, ci = forecasts[(t, best_label)]

    fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), gridspec_kw={"width_ratios": [2, 1]})
    ax = axes[0]
    ax.plot(train.index, train[t], label="Train (1998~2019)")
    ax.plot(test.index, test[t], label="Actual (2023~2024)", alpha=0.7)
    ax.plot(test.index, mean, "--", label=f"Forecast · {best_label}")
    ax.fill_between(test.index, ci[:, 0], ci[:, 1], alpha=0.2)
    ax.set_title(f"{t} · log(monthly arrivals)"); ax.set_xlabel("Year"); ax.legend()

    ax = axes[1]                                   # 평가 구간을 실제 인원으로 확대
    ax.plot(test.index, np.exp(test[t]), "o-", ms=3, label="Actual")
    ax.plot(test.index, np.exp(mean), "s--", ms=3, label="Forecast")
    ax.set_title(f"{t} · 2023-2024 arrivals (persons)"); ax.tick_params(axis="x", rotation=45); ax.legend()

    fig.tight_layout()
    fig.savefig(f"{FIG_DIR}/{t}_forecast.png", bbox_inches="tight")
    plt.show()

## 6. 결과 정리

`results/summary.csv`에 목적별·모델별 AIC, 환율 계수·p-value, 로그/실제 인원 기준 오차가 저장됩니다.
`figures/`에 ACF·PACF와 예측 그래프가 저장됩니다.